In [1]:
# Setup
import math_toolkit
math_toolkit.activate()

# The wave equation by Fourier modes

The wave equation describes a disturbance that moves without forgetting its shape:

$$
u_{tt}=c^2u_{xx}.
$$

On a periodic interval, Fourier modes make this equation diagonal. Each spatial frequency evolves like its own harmonic oscillator. The only thing the modes share is the same clock.

This worksheet solves the periodic wave equation by Fourier coefficients. First it animates three clean solutions, then it repeats the same figures with the individual Fourier modes drawn in green.


## The modal solution

For a 1-periodic solution, write the initial displacement and velocity as

$$
f(x)=a_0+\sum_{n=1}^{\infty}\left(a_n\cos(2\pi n x)+b_n\sin(2\pi n x)\right),
$$

and

$$
g(x)=\alpha_0+\sum_{n=1}^{\infty}\left(\alpha_n\cos(2\pi n x)+\beta_n\sin(2\pi n x)\right).
$$

If $u(x,0)=f(x)$ and $u_t(x,0)=g(x)$, then each coefficient oscillates in time:

$$
u(x,t)=a_0+\alpha_0t+\sum_{n=1}^{\infty}\left[\cos(2\pi c n t)\left(a_n\cos(2\pi n x)+b_n\sin(2\pi n x)\right)+\frac{\sin(2\pi c n t)}{2\pi c n}\left(\alpha_n\cos(2\pi n x)+\beta_n\sin(2\pi n x)\right)\right].
$$

The wave equation does not damp the modes. It rotates their phases, so initial displacement travels, and initial velocity is integrated into displacement by the same modal clocks.


We are solving
$$
\frac{\partial^2 u}{\partial t^2}=c^2\frac{\partial^2 u}{\partial x^2}
$$
with periodic boundary conditions on $-\frac12 \le x < \frac12$.

In this notebook the plotted time variable $t$ is an ordinary `math_toolkit`
parameter. Press the play button beside the $t$ slider to run the
Python-owned animation API.

The clean solution figures use an `N` slider that reaches 180. The later mode
figures repeat those solutions and add a separate `M` slider from 1 to 30 for
the number of individual green Fourier modes.


## Rough initial data

First use a deliberately uneven initial displacement. It has a smooth low-frequency shape plus several small high-frequency wiggles.

Press play on the $t$ slider. The broad motion comes from low modes, while the wiggles ride along because the wave equation preserves modal size.

In [2]:
wave_speed = 1
mode_cutoff = Symbol("M")
rough_modes = [
    (1, 0.00, 0.45),
    (3, 0.20, -0.08),
    (9, 0.08, 0.16),
    (17, -0.06, 0.05),
    (28, 0.03, -0.04),
]

rough_initial = 0
rough_wave = 0
rough_mode_terms = []
for mode, cos_coefficient, sin_coefficient in rough_modes:
    spatial_mode = (
        cos_coefficient * cos(2 * pi * mode * x)
        + sin_coefficient * sin(2 * pi * mode * x)
    )
    time_mode = cos(2 * pi * wave_speed * mode * t) * spatial_mode
    rough_initial += spatial_mode
    rough_wave += time_mode
    rough_mode_terms.append((mode, time_mode))

fig1 = figure("Rough Fourier data under the wave equation", new=True)
fig1.view.home_range = {"x": (-1/2, 1/2), "y": (-1.05, 1.05)}
fig1.show()

fig1.parameters({
    t: {
        "value": 0.0,
        "min": 0.0,
        "max": 1.0,
        "step": 0.005,
        "animated": True,
        "animation_mode": "forward",
        "animation_rate_hz": 20,
        "animation_speed": 0.1,
    },
})

fig1.plot(
    rough_initial,
    x,
    name="initial shape",
    label=r"$u(x,0)$",
    style={"color": "#64748b", "width": 3, "dash": "dot"},
    samples=1201,
)
fig1.plot(
    rough_wave,
    x,
    name="wave solution",
    label=r"$u(x,t)$",
    style={"color": "#2563eb", "width": 3},
    samples=1201,
)


:::{admonition} Question
When the broad part of the wave returns, what has happened to the high-frequency wiggles? Do they fade away, or do they keep moving with the solution?
:::

## Localized triangle wave

A compact triangle is a cleaner way to see the structure of the solution. With zero initial velocity, the initial shape splits into two half-size copies: one travels left and one travels right. On a periodic interval they wrap around and meet again.

Here the coefficients are defined by the Fourier integrals, in the same symbolic-numeric style as the square-wave inversion notebook.

In [3]:
triangle_center = -Rational(3, 16)
triangle_width = Rational(9, 50)
Tri = Function("Tri")(x) @EqDef@ (
    If(Abs(x - triangle_center) < triangle_width)
    .Then(1 - Abs(x - triangle_center) / triangle_width)
    .Otherwise(0)
)

FT_a_Tri_0 = Integral(Tri(s), (s, -Rational(1, 2), Rational(1, 2)))
FT_a_Tri = Function("FT_a_Tri", latex=r"a")(n) @EqDef@ (
    2 * Integral(
        Tri(s) * cos(2 * pi * n * s),
        (s, -Rational(1, 2), Rational(1, 2)),
    )
)
FT_b_Tri = Function("FT_b_Tri", latex=r"b")(n) @EqDef@ (
    2 * Integral(
        Tri(s) * sin(2 * pi * n * s),
        (s, -Rational(1, 2), Rational(1, 2)),
    )
)

W_Tri = Function("W_Tri")(x, t, N) @EqDef@ (
    FT_a_Tri_0
    + Sum(
        cos(2 * pi * wave_speed * n * t)
        * (
            FT_a_Tri(n) * cos(2 * pi * n * x)
            + FT_b_Tri(n) * sin(2 * pi * n * x)
        ),
        (n, 1, N),
    )
)
W_Tri_Mode = Function("W_Tri_Mode")(x, t, n) @EqDef@ (
    cos(2 * pi * wave_speed * n * t)
    * (
        FT_a_Tri(n) * cos(2 * pi * n * x)
        + FT_b_Tri(n) * sin(2 * pi * n * x)
    )
)

r"""
The triangle-wave solution below uses integral-defined Fourier coefficients
and the wave factor $\cos(2\pi c n t)$.
""" >> md

fig2 = figure("Localized triangle Fourier solution", new=True)
fig2.view.home_range = {"x": (-1/2, 1/2), "y": (-0.15, 1.1)}
fig2.show()

fig2.parameters({
    t: {
        "value": 0.0,
        "min": 0.0,
        "max": 10.0,
        "step": 0.005,
        "animated": True,
        "animation_mode": "forward",
        "animation_rate_hz": 20,
        "animation_speed": 0.1,
    },
    N: {"value": 80, "min": 1, "max": 180, "step": 1, "animated": False},
})

fig2.plot(
    Tri(x),
    x,
    name="initial triangle",
    label=r"$u(x,0)$",
    style={"color": "#64748b", "width": 3, "dash": "dot"},
    samples=2001,
)
fig2.plot(
    W_Tri(x, t, N),
    x,
    name="wave solution",
    label=r"$u_N(x,t)$",
    style={"color": "#dc2626", "width": 3},
    samples=2001,
)



The triangle-wave solution below uses integral-defined Fourier coefficients
and the wave factor $\cos(2\pi c n t)$.


:::{admonition} Question
At about $t=0.25$, where are the two traveling copies of the triangle? What happens when the two copies meet after wrapping around the periodic interval?
:::

## Localized Haar square-wave initial velocity

Now set the initial displacement to zero and give the string a localized Haar-style square-wave velocity supported on $[0.25,0.5]$: negative on the left half of that interval and positive on the right half. The solution starts flat, then the velocity profile immediately creates displacement through the sine-in-time part of the Fourier solution.

The gray dotted line is $u(x,0)=0$, the orange dashed curve is the initial velocity profile $v(x,0)=u_t(x,0)$, and the blue curve is the displacement $u(x,t)$.


In [4]:
VelSq = Function("VelSq")(x) @EqDef@ (10*
    If((x >= Rational(1, 4)) & (x < Rational(3, 8))).Then(-1)
    .If((x >= Rational(3, 8)) & (x <= Rational(1, 2))).Then(1)
    .Otherwise(0)
)

FT_a_VelSq_0 = Integral(VelSq(s), (s, -Rational(1, 2), Rational(1, 2)))
FT_a_VelSq = Function("FT_a_VelSq", latex=r"\alpha")(n) @EqDef@ (
    2 * Integral(
        VelSq(s) * cos(2 * pi * n * s),
        (s, -Rational(1, 2), Rational(1, 2)),
    )
)
FT_b_VelSq = Function("FT_b_VelSq", latex=r"\beta")(n) @EqDef@ (
    2 * Integral(
        VelSq(s) * sin(2 * pi * n * s),
        (s, -Rational(1, 2), Rational(1, 2)),
    )
)

W_VelSq = Function("W_VelSq")(x, t, N) @EqDef@ (
    FT_a_VelSq_0 * t
    + Sum(
        sin(2 * pi * wave_speed * n * t)
        / (2 * pi * wave_speed * n)
        * (
            FT_a_VelSq(n) * cos(2 * pi * n * x)
            + FT_b_VelSq(n) * sin(2 * pi * n * x)
        ),
        (n, 1, N),
    )
)
W_VelSq_Mode = Function("W_VelSq_Mode")(x, t, n) @EqDef@ (
    sin(2 * pi * wave_speed * n * t)
    / (2 * pi * wave_speed * n)
    * (
        FT_a_VelSq(n) * cos(2 * pi * n * x)
        + FT_b_VelSq(n) * sin(2 * pi * n * x)
    )
)

r"""
The localized Haar velocity solution has zero initial displacement. Its Fourier
velocity coefficients enter through $\sin(2\pi c n t)/(2\pi c n)$.
""" >> md

fig3 = figure("Localized Haar initial velocity Fourier solution", new=True)
fig3.view.home_range = {"x": (-1/2, 1/2), "y": (-1.2, 1.2)}
fig3.show()

fig3.parameters({
    t: {
        "value": 0.0,
        "min": 0.0,
        "max": 1.0,
        "step": 0.01,
        "animated": True,
        "animation_mode": "forward",
        "animation_rate_hz": 20,
        "animation_speed": 0.18,
    },
    N: {"value": 80, "min": 1, "max": 180, "step": 1, "animated": False},
})

fig3.plot(
    0 * x,
    x,
    name="initial displacement",
    label=r"$u(x,0)=0$",
    style={"color": "#64748b", "width": 3, "dash": "dot"},
    samples=2001,
)
fig3.plot(
    VelSq(x)/10,
    x,
    name="initial velocity",
    label=r"$v(x,0)/10$",
    style={"color": "#f97316", "width": 3, "dash": "dash"},
    samples=2001,
)
fig3.plot(
    W_VelSq(x, t, N),
    x,
    name="wave solution",
    label=r"$u_N(x,t)$",
    style={"color": "#2563eb", "width": 3},
    samples=2001,
)



The localized Haar velocity solution has zero initial displacement. Its Fourier
velocity coefficients enter through $\sin(2\pi c n t)/(2\pi c n)$.


:::{admonition} Question
At $t=0$, the displacement is flat but the velocity is not. After pressing play, how does the localized negative-left and positive-right velocity profile begin to separate the displacement?
:::


## The same animations with individual Fourier modes

The next three figures repeat the clean animations above. The main solution still uses an `N` cutoff up to 180, while the separate `M` slider controls how many individual green Fourier modes are drawn, from 1 to 30 with default 3.


In [9]:
fig4 = figure("Rough Fourier data with individual modes", new=True)
fig4.view.home_range = {"x": (-1/2, 1/2), "y": (-1.05, 1.05)}
fig4.show()

fig4.parameters({
    t: {
        "value": 0.0,
        "min": 0.0,
        "max": 1.0,
        "step": 0.005,
        "animated": True,
        "animation_mode": "forward",
        "animation_rate_hz": 20,
        "animation_speed": 0.1,
    },
    mode_cutoff: {"value": 3, "min": 1, "max": 30, "step": 1, "animated": False},
})

fig4.plot(
    rough_initial,
    x,
    name="initial shape",
    label=r"$u(x,0)$",
    style={"color": "#64748b", "width": 3, "dash": "dot"},
    samples=1201,
)
for mode, time_mode in rough_mode_terms:
    fig4.plot(
        If(mode_cutoff >= mode).Then(time_mode).Otherwise(0),
        x,
        name=f"mode {mode}",
        label=rf"$n={mode}$",
        style={"color": "#16a34a", "width": 2, "opacity": 0.35},
        samples=200,
    )
fig4.plot(
    rough_wave,
    x,
    name="wave solution",
    label=r"$u(x,t)$",
    style={"color": "#2563eb", "width": 3},
    samples=1201,
)

fig5 = figure("Localized triangle with individual modes", new=True)
fig5.view.home_range = {"x": (-1/2, 1/2), "y": (-0.15, 1.1)}
fig5.show()

fig5.parameters({
    t: {
        "value": 0.0,
        "min": 0.0,
        "max": 10.0,
        "step": 0.005,
        "animated": True,
        "animation_mode": "forward",
        "animation_rate_hz": 20,
        "animation_speed": 0.1,
    },
    N: {"value": 80, "min": 1, "max": 180, "step": 1, "animated": False},
    mode_cutoff: {"value": 3, "min": 1, "max": 30, "step": 1, "animated": False},
})

fig5.plot(
    Tri(x),
    x,
    name="initial triangle",
    label=r"$u(x,0)$",
    style={"color": "#64748b", "width": 3, "dash": "dot"},
    samples=2001,
)
for mode in range(1, 31):
    fig5.plot(
        If(mode_cutoff >= mode).Then(W_Tri_Mode(x, t, mode)).Otherwise(0),
        x,
        name=f"mode {mode}",
        label=rf"$n={mode}$",
        style={"color": "#16a34a", "width": 2, "opacity": 0.35},
        samples=200,
    )
fig5.plot(
    W_Tri(x, t, N),
    x,
    name="wave solution",
    label=r"$u_N(x,t)$",
    style={"color": "#dc2626", "width": 3},
    samples=2001,
)

fig6 = figure("Localized Haar initial velocity with individual modes", new=True)
fig6.view.home_range = {"x": (-1/2, 1/2), "y": (-1.2, 1.2)}
fig6.show()

fig6.parameters({
    t: {
        "value": 0.0,
        "min": 0.0,
        "max": 1.0,
        "step": 0.005,
        "animated": True,
        "animation_mode": "forward",
        "animation_rate_hz": 20,
        "animation_speed": 0.1,
    },
    N: {"value": 80, "min": 1, "max": 180, "step": 1, "animated": False},
    mode_cutoff: {"value": 3, "min": 1, "max": 30, "step": 1, "animated": False},
})

fig6.plot(
    0 * x,
    x,
    name="initial displacement",
    label=r"$u(x,0)=0$",
    style={"color": "#64748b", "width": 3, "dash": "dot"},
    samples=2001,
)
fig6.plot(
    VelSq(x)/10,
    x,
    name="initial velocity",
    label=r"$v(x,0)/10$",
    style={"color": "#f97316", "width": 3, "dash": "dash"},
    samples=2001,
)
for mode in range(1, 31):
    fig6.plot(
        If(mode_cutoff >= mode).Then(W_VelSq_Mode(x, t, mode)).Otherwise(0),
        x,
        name=f"mode {mode}",
        label=rf"$n={mode}$",
        style={"color": "#16a34a", "width": 2, "opacity": 0.35},
        samples=200,
    )
fig6.plot(
    W_VelSq(x, t, N),
    x,
    name="wave solution",
    label=r"$u_N(x,t)$",
    style={"color": "#2563eb", "width": 3},
    samples=2001,
)


## Now you try

Change `wave_speed` or the time slider's `animation_speed` in one of the animation cells. Larger values make the wave travel faster. Smaller values make the same movie feel closer to slow motion.

Then change the initial data. For example, add another high-frequency sine or cosine mode to `rough_modes`, move the triangle by changing `triangle_center`, shift the localized Haar velocity interval, or adjust the Fourier cutoffs `N` and `M` and watch what changes.


In [6]:
# Experiment with your own initial displacement or velocity here.
